# 30-second check v3: a better second detector, chosen on MIT-BIH, tested on CinC 2017 (Notebook 9)
**Bektur Asilov · COMSE-23 · Ala-Too**

Notebook 8: the two-detector gate (v2) catches noise, but 44 % of clean normal checks needed a repeat because the
simple second detector disagrees with the main one. Here the second detector is improved **without touching CinC 2017**:

1. **Selection on MIT-BIH Arrhythmia** (48 records, cardiologist beat labels). 16 settings of the second detector
   (smoothing 1 or 3 samples; baseline window 0.6 s or 0.12 s; threshold 0.5 or 0.35 of the peak; refractory
   0.25 or 0.30 s). The one with the **highest F1** (150 ms, whole records) is chosen; on a tie the earlier one in
   the list (the current v2 setting is first). This rule is fixed before the run.
2. **Test on CinC 2017** exactly as in Notebook 8 (first 30 s; clean, and AF/Normal with NSTDB em/ma at 12/6/0 dB;
   same seeds). Reported: v1 (no gate), v2 (current gate), v3 (gate with the chosen detector). bSQI threshold
   0.8 is not changed.

The server code below is pasted unchanged. Kaggle: Internet **On**, CPU.

In [ ]:
import os, time, json, subprocess, sys, zipfile, zlib, itertools
import numpy as np, pandas as pd
from scipy.signal import butter, sosfilt, resample_poly
from multiprocessing import Pool
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
FS, SEED = 125, 42
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
CINC_DIR, NS_DIR, MIT_DIR = os.environ.get('CINC_DIR'), os.environ.get('NS_DIR'), os.environ.get('MITDB_DIR')
def fetch(files, dst, bases):
    os.makedirs(dst, exist_ok=True)
    for f in files:
        for base in bases:
            if subprocess.run(['curl', '-fsSL', '--max-time', '1200', '-o', f'{dst}/{f}', base + f]).returncode == 0: break
        else: raise RuntimeError('download failed: ' + f)
MREC = [100,101,102,103,104,105,106,107,108,109,111,112,113,114,115,116,117,118,119,121,122,123,124,
        200,201,202,203,205,207,208,209,210,212,213,214,215,217,219,220,221,222,223,228,230,231,232,233,234]
if not MIT_DIR:
    MIT_DIR = '/kaggle/tmp/mitdb'
    fetch([f'{r}.{e}' for r in MREC for e in ('hea', 'dat', 'atr')], MIT_DIR, ['https://physionet-open.s3.amazonaws.com/mitdb/1.0.0/', 'https://physionet.org/files/mitdb/1.0.0/'])
if not CINC_DIR:
    fetch(['REFERENCE-v3.csv', 'training2017.zip'], '/kaggle/tmp', ['https://physionet-open.s3.amazonaws.com/challenge-2017/1.0.0/', 'https://physionet.org/files/challenge-2017/1.0.0/'])
    zipfile.ZipFile('/kaggle/tmp/training2017.zip').extractall('/kaggle/tmp')
    CINC_DIR, REF = '/kaggle/tmp/training2017', '/kaggle/tmp/REFERENCE-v3.csv'
else:
    REF = os.path.join(CINC_DIR, 'REFERENCE-v3.csv')
if not NS_DIR:
    NS_DIR = '/kaggle/tmp/nstdb'
    fetch([f'{r}.{e}' for r in ('em', 'ma') for e in ('hea', 'dat')], NS_DIR, ['https://physionet.org/files/nstdb/1.0.0/', 'https://physionet-open.s3.amazonaws.com/nstdb/1.0.0/'])
ref = pd.read_csv(REF, header=None, names=['record', 'label'])
NOISE = {k: (lambda r: (r.p_signal[:, 0], r.fs))(wfdb.rdrecord(os.path.join(NS_DIR, k), channels=[0])) for k in ('em', 'ma')}
MREC = [r for r in MREC if os.path.exists(f'{MIT_DIR}/{r}.dat')]
print(len(ref), 'CinC recordings,', len(MREC), 'MIT-BIH records')

## Server code, unchanged (`backend/app/ecgtools.py` without the PNG drawing)

In [ ]:
"""ECG strip analysis on the server: R-peaks, RR intervals, Poincaré points, signal quality index, PNG image.

Pure Python (strips are short: 10 s x 125 Hz). The R-peak detector is the firmware algorithm
(derivative^2 -> 150 ms moving average -> adaptive threshold, 250 ms refractory), adapted to run offline
on a short strip: the threshold tracker is primed with the first 2 s instead of a 2-s warm-up.
"""

import io
import math
import statistics


def _highpass(x: list[float], fs: int) -> list[float]:
    """Remove baseline wander: subtract a 0.6-s centred moving mean."""
    w = max(3, int(0.6 * fs)) | 1
    h = w // 2
    n = len(x)
    pre = [0.0]
    for v in x:
        pre.append(pre[-1] + v)
    out = []
    for i in range(n):
        a, b = max(0, i - h), min(n, i + h + 1)
        out.append(x[i] - (pre[b] - pre[a]) / (b - a))
    return out


def r_peaks(x: list[float], fs: int = 125) -> list[int]:
    n = len(x)
    if n < fs:
        return []
    y = _highpass(x, fs)
    ma = max(3, int(0.15 * fs))
    env, buf, s, prev = [], [0.0] * ma, 0.0, y[0]
    for i, v in enumerate(y):
        e = (v - prev) ** 2
        prev = v
        s += e - buf[i % ma]
        buf[i % ma] = e
        env.append(s / ma)
    decay = 0.5 ** (1 / (2.0 * fs))
    thr, refr = 0.35, int(0.25 * fs)
    pk = max(env[: min(n, 2 * fs)]) or 1e-12
    out, last, armed = [], -10 ** 9, True
    for i in range(int(0.2 * fs), n):
        pk = max(pk * decay, env[i])
        if armed and env[i] > thr * pk and i - last > refr:
            # the envelope lags the QRS by ~MA/2: take the largest |amplitude| just before
            a, b = max(0, i - ma - 4), min(n, i + 3)
            j = max(range(a, b), key=lambda k: abs(y[k]))
            if not out or j - out[-1] > refr:
                out.append(j)
            last, armed = i, False
        elif not armed and env[i] < 0.5 * thr * pk:
            armed = True
    return out


def rr_intervals(peaks: list[int], fs: int = 125) -> list[float]:
    return [round((b - a) / fs, 3) for a, b in zip(peaks, peaks[1:])]


def poincare(rr: list[float]) -> dict:
    """Points (RRn, RRn+1) and the SD1/SD2 descriptors (SD1 = short-term, beat-to-beat variability)."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    return poincare_points([[a, b] for a, b in zip(rr, rr[1:])])


def poincare_points(pts: list[list[float]]) -> dict:
    if len(pts) < 3:
        return {"points": pts, "sd1": None, "sd2": None, "ratio": None}
    d1 = [(b - a) / math.sqrt(2) for a, b in pts]
    d2 = [(b + a) / math.sqrt(2) for a, b in pts]
    sd1, sd2 = statistics.pstdev(d1), statistics.pstdev(d2)
    return {"points": pts, "sd1": round(sd1 * 1000, 1), "sd2": round(sd2 * 1000, 1),
            "ratio": round(sd1 / sd2, 2) if sd2 > 0 else None}


def irregularity(rr: list[float]) -> float | None:
    """Mean |successive difference| / mean RR — high (>~0.1) in AF, low in sinus rhythm."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    if len(rr) < 4:
        return None
    return round(statistics.mean(abs(b - a) for a, b in zip(rr, rr[1:])) / statistics.mean(rr), 3)


def quality(x: list[float], fs: int = 125, peaks: list[int] | None = None) -> dict:
    """Signal quality index 0..1 from four checks a cardiologist does by eye:
    no flat line / clipping, plausible heart rate, consistent QRS shape, little high-frequency noise."""
    n = len(x)
    if n < 2 * fs:
        return {"sqi": 0.0, "level": "poor", "reasons": ["too_short"]}
    peaks = r_peaks(x, fs) if peaks is None else peaks
    y = _highpass(x, fs)
    reasons = []
    rng = max(y) - min(y)
    flat = sum(1 for a, b in zip(y, y[1:]) if abs(b - a) < 1e-4) / n
    s_flat = 0.0 if rng < 0.05 or flat > 0.5 else 1.0
    if s_flat == 0:
        reasons.append("flat")
    secs = n / fs
    bpm = len(peaks) / secs * 60
    s_rate = 1.0 if 30 <= bpm <= 200 else 0.0
    if not s_rate:
        reasons.append("rate")
    # QRS template correlation
    h = int(0.1 * fs)
    segs = [y[p - h:p + h] for p in peaks if p - h >= 0 and p + h <= n]
    if len(segs) >= 3:
        tmpl = [statistics.median(col) for col in zip(*segs)]

        def corr(a, b):
            ma, mb = statistics.mean(a), statistics.mean(b)
            num = sum((u - ma) * (v - mb) for u, v in zip(a, b))
            den = math.sqrt(sum((u - ma) ** 2 for u in a) * sum((v - mb) ** 2 for v in b)) or 1e-12
            return num / den
        s_shape = max(0.0, statistics.mean(corr(s, tmpl) for s in segs))
    else:
        s_shape = 0.0
    if s_shape < 0.7:
        reasons.append("shape")
    # high-frequency noise: energy of the second difference relative to the signal
    d2 = [y[i + 1] - 2 * y[i] + y[i - 1] for i in range(1, n - 1)]
    noise = (sum(v * v for v in d2) / len(d2)) ** 0.5 / (statistics.pstdev(y) or 1e-12)
    s_noise = max(0.0, min(1.0, 1.5 - noise))
    if s_noise < 0.5:
        reasons.append("noise")
    sqi = s_flat * s_rate * (0.6 * s_shape + 0.4 * s_noise)
    sqi = round(max(0.0, min(1.0, sqi)), 2)
    return {"sqi": sqi, "level": "good" if sqi >= 0.7 else "fair" if sqi >= 0.4 else "poor", "reasons": reasons}


PAUSE_S = 3.0   # an RR gap of 3 s or more is a pause worth showing to a doctor


def pauses(peaks: list[int], fs: int = 125) -> list[dict]:
    return [{"at_s": round(a / fs, 2), "rr_s": round((b - a) / fs, 2)}
            for a, b in zip(peaks, peaks[1:]) if (b - a) / fs >= PAUSE_S]


def analyze(x: list[float], fs: int = 125) -> dict:
    p = r_peaks(x, fs)
    rr = rr_intervals(p, fs)
    q = quality(x, fs, p)
    # a pause only counts when the rest of the strip is clean; otherwise it is more likely a lost electrode
    return {"r_peaks": p, "rr": rr, "poincare": poincare(rr), "irregularity": irregularity(rr),
            "quality": q, "pauses": pauses(p, fs) if q["sqi"] >= 0.4 else []}


def _movmean(x: list[float], w: int) -> list[float]:
    """Centred moving mean over w samples (w odd), shrinking at the edges — the same as in _highpass."""
    h, n = w // 2, len(x)
    pre = [0.0]
    for v in x:
        pre.append(pre[-1] + v)
    return [(pre[min(n, i + h + 1)] - pre[max(0, i - h)]) / (min(n, i + h + 1) - max(0, i - h)) for i in range(n)]


# second detector settings (smooth samples, baseline window s, threshold fraction, refractory s); v2 = the first
AMP_DEFAULT = (1, 0.6, 0.5, 0.25)


def r_peaks_amp(x: list[float], fs: int = 125, params: tuple = AMP_DEFAULT) -> list[int]:
    """Second, independent R-peak detector for the agreement check: amplitude of the signal after removing
    the baseline (not the slope energy the main detector uses), adaptive threshold at a fraction of the
    decaying peak, refractory period. Optional light smoothing and a short baseline window make it a crude
    QRS band-pass that ignores T-waves better."""
    w1, w2s, frac, refr_s = params
    n = len(x)
    if n < fs:
        return []
    y = _movmean(x, w1) if w1 > 1 else list(x)
    w2 = max(3, int(w2s * fs)) | 1
    base = _movmean(y, w2)
    a = [abs(u - b) for u, b in zip(y, base)]
    decay = 0.5 ** (1 / (2.0 * fs))
    refr, look = int(refr_s * fs), int(0.1 * fs)
    pk = max(a[: min(n, 2 * fs)]) or 1e-12
    out, i = [], 0
    while i < n:
        pk = max(pk * decay, a[i])
        if a[i] > frac * pk and (not out or i - out[-1] > refr):
            j = max(range(i, min(n, i + look)), key=lambda k: a[k])
            if not out or j - out[-1] > refr:
                out.append(j)
            i = j + 1
            continue
        i += 1
    return out


def bsqi(p1: list[int], p2: list[int], fs: int = 125, tol_s: float = 0.15) -> float:
    """Agreement of two detectors (Li, Mark & Clifford 2008): matched / (n1 + n2 - matched), match within 150 ms."""
    if not p1 and not p2:
        return 0.0
    tol, used, m = int(tol_s * fs), set(), 0
    j = 0
    for p in p1:
        while j < len(p2) and p2[j] < p - tol:
            j += 1
        k = j
        while k < len(p2) and p2[k] <= p + tol:
            if k not in used:
                used.add(k)
                m += 1
                break
            k += 1
    return round(m / (len(p1) + len(p2) - m), 3)


# "Check me in 30 seconds" (wellness wording), see ml/CHECK30_CINC2017.md and ml/CHECK30_BSQI.md
# v1 (CinC 2017, never tuned on it): 91.4 % of AF recordings irregular, 14.1 % of normal ones too; noise not caught.
# v2 adds the two-detector agreement gate (bSQI >= 0.8), rules fixed before Notebook 8: false "irregular" on clean
# normal rhythm 14.1 % -> 3.3 %, noise caught (97.5 % of noisy recordings -> "could not read"), but 44 % of clean
# normal checks need a repeat and AF found drops 91.4 -> 69.4 %. See ml/CHECK30_BSQI.md.
CHECK_MIN_BEATS, CHECK_MIN_SQI, CHECK_IRR, CHECK_MIN_BSQI = 16, 0.4, 0.10, 0.8


def check(x: list[float], fs: int = 125, gate: bool = True) -> dict:
    """Verdict for a 30-s spot check: "regular", "irregular" (show a doctor) or "no_verdict" (repeat)."""
    a = analyze(x, fs)
    n, sqi, irr = len(a["r_peaks"]), a["quality"]["sqi"], a["irregularity"]
    b = bsqi(a["r_peaks"], r_peaks_amp(x, fs), fs)
    if n < CHECK_MIN_BEATS or sqi < CHECK_MIN_SQI or irr is None or (gate and b < CHECK_MIN_BSQI):
        verdict = "no_verdict"
    else:
        verdict = "irregular" if irr > CHECK_IRR else "regular"
    bpm = round(len(a["r_peaks"]) / (len(x) / fs) * 60) if x else None
    return {"verdict": verdict, "beats": n, "sqi": sqi, "irregularity": irr, "bsqi": b, "bpm": bpm,
            "reasons": a["quality"]["reasons"] + (["detectors_disagree"] if gate and b < CHECK_MIN_BSQI else [])}

## 1. Choose the second detector on MIT-BIH Arrhythmia (CinC 2017 is not used here)

In [ ]:
def to_125(sig, fs):
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)
BEAT = set('NLRBAaJSVrFejnE/fQ?')
CANDS = [AMP_DEFAULT] + [c for c in itertools.product((1, 3), (0.6, 0.12), (0.5, 0.35), (0.25, 0.30)) if c != AMP_DEFAULT]
def f1(ref_, det, tol):
    j = np.searchsorted(det, ref_); used = set(); tp = 0
    for r, i in zip(ref_, j):
        best = None
        for k in (i - 1, i):
            if 0 <= k < len(det) and k not in used and abs(det[k] - r) <= tol and (best is None or abs(det[k] - r) < abs(det[best] - r)): best = k
        if best is not None: used.add(best); tp += 1
    return tp, len(ref_) - tp, len(det) - tp
def mit_job(args):
    rec, ci = args
    r = wfdb.rdrecord(f'{MIT_DIR}/{rec}', channels=[0]); ann = wfdb.rdann(f'{MIT_DIR}/{rec}', 'atr')
    x = list(map(float, to_125(r.p_signal[:, 0], r.fs)))
    ref_ = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in BEAT]) * FS / r.fs
    det = np.array(r_peaks_amp(x, FS, CANDS[ci]), float)
    tp, fn, fp = f1(ref_, det, 0.15 * FS)
    return dict(record=rec, cand=ci, TP=tp, FN=fn, FP=fp)
T0 = time.time()
with Pool(os.cpu_count()) as pool:
    M = pd.DataFrame(pool.map(mit_job, [(r, c) for r in MREC for c in range(len(CANDS))], chunksize=4))
G = M.groupby('cand')[['TP', 'FN', 'FP']].sum()
G['sens'] = G.TP / (G.TP + G.FN); G['ppv'] = G.TP / (G.TP + G.FP); G['F1'] = 2 * G.TP / (2 * G.TP + G.FN + G.FP)
G['params(smooth, base_s, frac, refr_s)'] = [CANDS[i] for i in G.index]
print(G.round(4).to_string()); print(f'{time.time()-T0:.0f} s')
CHOSEN = CANDS[int(G.F1.round(6).idxmax())]          # idxmax returns the first maximum: ties go to the earlier one
print('chosen second detector:', CHOSEN, '| v2 setting was', AMP_DEFAULT)

## 2. Test on CinC 2017: v1 (no gate), v2 (current gate), v3 (gate with the chosen detector)

In [ ]:
def bandpass(x, fs): return sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), x)
def noise_for(kind, n, fs, rng):
    z, nfs = NOISE[kind]
    if nfs != fs:
        from math import gcd
        g = gcd(int(fs), int(nfs)); z = resample_poly(z, int(fs) // g, int(nfs) // g)
    s = rng.randint(0, len(z) - n); return z[s:s + n]
CONDS = [('clean', None)] + [(k, s) for k in ('em', 'ma') for s in (12, 6, 0)]
def job(args):
    rec, lab, kind, snr = args
    r = wfdb.rdrecord(os.path.join(CINC_DIR, rec)); x = r.p_signal[:30 * r.fs, 0]
    if kind != 'clean':
        rng = np.random.RandomState((SEED * 1000003 + zlib.crc32(f"{rec}{kind}{snr}".encode())) % 2**31)
        z = noise_for(kind, len(x), r.fs, rng)
        ps, pn = bandpass(x, r.fs).var(), bandpass(z, r.fs).var()
        x = x + z * np.sqrt(ps / (pn * 10 ** (snr / 10)) if pn > 0 else 0)
    x = list(map(float, to_125(x, r.fs)))
    c = check(x, FS, gate=False)
    main = r_peaks(x, FS)
    b3 = bsqi(main, r_peaks_amp(x, FS, CHOSEN), FS)
    gated = lambda b: 'no_verdict' if c['verdict'] != 'no_verdict' and b < CHECK_MIN_BSQI else c['verdict']
    return dict(record=rec, label=lab, noise=kind, snr=snr if snr is not None else 99, v1=c['verdict'],
                v2=gated(c['bsqi']), v3=gated(b3), bsqi2=c['bsqi'], bsqi3=b3)
tasks = [(r, l, 'clean', None) for r, l in zip(ref.record, ref.label)]
tasks += [(r, l, k, s) for k, s in CONDS[1:] for r, l in zip(ref.record, ref.label) if l in ('A', 'N')]
T0 = time.time()
with Pool(os.cpu_count()) as pool:
    rows = []
    for i, row in enumerate(pool.imap_unordered(job, tasks, chunksize=64)):
        rows.append(row)
        if (i + 1) % 5000 == 0: print(f'{i+1}/{len(tasks)} [{time.time()-T0:.0f} s]', flush=True)
D = pd.DataFrame(rows); D.to_csv(f'{OUT_DIR}/check30_v3_predictions.csv', index=False)
print('done', len(D), f'in {time.time()-T0:.0f} s')

## Results

In [ ]:
def shares(g, col):
    v = g[col].value_counts(normalize=True)
    return pd.Series({'irregular': v.get('irregular', 0), 'no_verdict': v.get('no_verdict', 0), 'regular': v.get('regular', 0)})
pd.set_option('display.width', 220)
C = D[D.noise == 'clean']
S = {'chosen': list(CHOSEN), 'mitdb': G.reset_index().round(5).to_dict('records'), 'clean': {}}
for lab in ('A', 'N', 'O', '~'):
    g = C[C.label == lab]; S['clean'][lab] = {col: shares(g, col).round(4).to_dict() for col in ('v1', 'v2', 'v3')}
    print(lab, {col: shares(g, col).round(3).to_dict() for col in ('v1', 'v2', 'v3')})
T = []
for (k, s), g in D[D.noise != 'clean'].groupby(['noise', 'snr']):
    for col in ('v1', 'v2', 'v3'):
        a, n = shares(g[g.label == 'A'], col), shares(g[g.label == 'N'], col)
        T.append(dict(noise=k, snr=s, rule=col, AF_irregular=a.irregular, AF_no_verdict=a.no_verdict,
                      N_irregular=n.irregular, N_no_verdict=n.no_verdict, N_regular=n.regular))
T = pd.DataFrame(T).sort_values(['noise', 'snr', 'rule'], ascending=[True, False, True]); print(T.round(3).to_string(index=False))
S['noise'] = T.round(4).to_dict('records')
json.dump(S, open(f'{OUT_DIR}/check30_v3_summary.json', 'w'), indent=1, default=float)